# Experiment 17 — Theme Business Needs + Description + Stage — Individual Calls

No record key is sent to the model.

## What the LLM sees

Theme Business Needs + Theme Description + Stage data + candidate L3s.

**Not sent:** record key, ground truth, L1/L2 hierarchy.

## Configuration and data

In [ ]:
from pathlib import Path
from time import perf_counter
import ast, json, os
import pandas as pd
from IPython.display import display
from common import call_llm_with_metrics, load_gateway, parse_json_response, save_results_excel, score_sets

HERE=Path.cwd()
def resolve(name, env):
    if os.getenv(env): return Path(os.environ[env]).expanduser()
    for root in (HERE,HERE.parent,HERE/'l3_experiments'):
        p=root/name
        if p.exists(): return p
    raise FileNotFoundError(name)
DATASET_PATH=resolve('golden_valid_set.parquet','L3_GOLDEN_VALID_SET_PATH')
GT_PATH=resolve('results/epic_l3_ground_truth_full_golden.xlsx','L3_GROUND_TRUTH_PATH')
SAMPLE_SIZE=50
SAMPLE_SEED=42

EXPERIMENT_NAME='E17_THEME_NEEDS_DESCRIPTION_STAGE_INDIVIDUAL_NO_RECORD_KEY'


## Load combined valid set and Jira GT

In [ ]:
def clean(v):
    if v is None: return ''
    try:
        if pd.isna(v): return ''
    except (TypeError,ValueError): pass
    return str(v).strip()

def parsed(v):
    if v is None: return None
    if hasattr(v,'tolist') and not isinstance(v,(str,bytes)): v=v.tolist()
    if isinstance(v,(dict,list,tuple,set)): return v
    try:
        if pd.isna(v): return None
    except (TypeError,ValueError): pass
    s=str(v).strip()
    if not s: return None
    for fn in (json.loads,ast.literal_eval):
        try: return fn(s)
        except Exception: pass
    return s

def as_list(v):
    v=parsed(v)
    if v is None: return []
    if isinstance(v,(list,tuple,set)): return [clean(x) for x in v if clean(x)]
    return [clean(v)] if clean(v) else []

def col(df,*names,required=True):
    lower={str(c).lower():c for c in df.columns}
    for n in names:
        if n in df.columns: return n
        if n.lower() in lower: return lower[n.lower()]
    if required: raise KeyError(f'Missing column; expected one of {names}')

def gt_map():
    g=pd.read_excel(GT_PATH,sheet_name='jira_l3_ground_truth',dtype=str)
    k,l=col(g,'epic_key'),col(g,'l3_capability_id')
    s=col(g,'status',required=False)
    if s: g=g[g[s].fillna('').str.lower().eq('ok')]
    g=g[g[l].notna()].copy(); g[l]=g[l].map(clean)
    return {clean(k0):sorted(set(x[l])) for k0,x in g.groupby(k,sort=False)}

def load_population():
    f=pd.read_parquet(DATASET_PATH)
    req={
      'theme_key':col(f,'theme_key','key'),'record_key':col(f,'epic_key'),
      'stage_ids':col(f,'stage_ids'),'candidate_l3_ids':col(f,'candidate_l3_ids'),
      'theme_description':col(f,'theme_description','description'),
      'theme_business_needs':col(f,'theme_business_needs','businessNeeds')}
    opt={
      'stage_data':col(f,'stage_data','value_stream_stages','stage_contexts',required=False),
      'candidate_l3_capabilities':col(f,'candidate_l3_capabilities','candidate_l3s',required=False),
      'candidate_l3_by_stage':col(f,'candidate_l3_by_stage','stage_candidate_l3_ids','stage_candidate_l3_capabilities',required=False)}
    p=pd.DataFrame({k:f[v] for k,v in req.items()})
    for k,v in opt.items(): p[k]=f[v] if v else None
    for k in ('theme_key','record_key','theme_description','theme_business_needs'): p[k]=p[k].map(clean)
    p=p.drop_duplicates(['theme_key','record_key']).sort_values(['theme_key','record_key']).reset_index(drop=True)
    if len(p)<SAMPLE_SIZE: raise ValueError(f'Need {SAMPLE_SIZE} rows, found {len(p)}')
    p=p.sample(SAMPLE_SIZE,random_state=SAMPLE_SEED,replace=False)
    gm=gt_map(); p['gt_l3_ids']=p['record_key'].map(gm)
    miss=p[p.gt_l3_ids.isna()].record_key.tolist()
    if miss: raise ValueError(f'GT workbook missing sampled records: {miss}')
    return p.sort_values(['theme_key','record_key']).reset_index(drop=True)

evaluation_population=load_population()
print(f'Selected {len(evaluation_population)} rows from golden_valid_set.parquet with seed={SAMPLE_SEED} across {evaluation_population.theme_key.nunique()} Themes.')
display(evaluation_population.head(50))


## Build Stage payloads

In [ ]:
def candidate(x):
    if isinstance(x,dict):
        cid=clean(x.get('capability_id') or x.get('l3_capability_id') or x.get('id'))
        if not cid: return None
        y={'capability_id':cid}
        for out,keys in {'capability_name':('capability_name','l3_capability_name','name'),'capability_description':('capability_description','description'),'capability_tier':('capability_tier','tier')}.items():
            v=next((x.get(k) for k in keys if x.get(k)),None)
            if clean(v): y[out]=clean(v)
        return y
    return {'capability_id':clean(x)} if clean(x) else None

def candidates(v):
    v=parsed(v)
    if v is None: return []
    if isinstance(v,dict) and any(k in v for k in ('capability_id','l3_capability_id','id')): v=[v]
    if not isinstance(v,(list,tuple,set)): v=[v]
    out=[]; seen=set()
    for x in v:
        y=candidate(x)
        if y and y['capability_id'] not in seen: seen.add(y['capability_id']); out.append(y)
    return out

def stage(x,fallback=''):
    if isinstance(x,dict):
        sid=clean(x.get('stage_id') or x.get('value_stream_stage_id') or x.get('id') or fallback)
        if not sid: return None
        y={'stage_id':sid}
        for out,keys in {'stage_name':('stage_name','name'),'stage_description':('stage_description','description'),'entrance_criteria':('entrance_criteria',),'exit_criteria':('exit_criteria',)}.items():
            v=next((x.get(k) for k in keys if x.get(k)),None)
            if clean(v): y[out]=clean(v)
        return y
    sid=clean(x or fallback); return {'stage_id':sid} if sid else None

def row_stages(r):
    ids=as_list(r['stage_ids']); details={sid:{'stage_id':sid} for sid in ids}
    sd=parsed(r.get('stage_data'))
    if isinstance(sd,dict):
        if any(k in sd for k in ('stage_id','value_stream_stage_id','id')): sd=[sd]
        else: sd=[stage(v,k) for k,v in sd.items()]
    if isinstance(sd,(list,tuple,set)):
        for x in sd:
            y=x if isinstance(x,dict) and 'stage_id' in x else stage(x)
            if y and y['stage_id'] in details: details[y['stage_id']].update(y)
    by={sid:[] for sid in ids}; m=parsed(r.get('candidate_l3_by_stage'))
    if isinstance(m,dict):
        for sid in ids:
            if sid in m: by[sid]=candidates(m[sid])
    rich=parsed(r.get('candidate_l3_capabilities'))
    if isinstance(rich,(list,tuple)):
        staged={}; loose=[]
        for x in rich:
            sid=clean(x.get('stage_id') or x.get('value_stream_stage_id')) if isinstance(x,dict) else ''
            (staged.setdefault(sid,[]).append(x) if sid else loose.append(x))
        for sid in ids:
            if not by[sid] and sid in staged: by[sid]=candidates(staged[sid])
        if loose:
            loose=candidates(loose)
            for sid in ids:
                if not by[sid]: by[sid]=loose
    base=candidates(r['candidate_l3_ids'])
    for sid in ids:
        if not by[sid]: by[sid]=base
    return [{'stage':details[sid],'candidates':by[sid]} for sid in ids]

def merged_stages(rows):
    m={}
    for r in rows.to_dict('records'):
        for p in row_stages(r):
            sid=p['stage']['stage_id']; t=m.setdefault(sid,{'stage':{'stage_id':sid},'candidates':{}})
            for k,v in p['stage'].items():
                if v and not t['stage'].get(k): t['stage'][k]=v
            for c in p['candidates']:
                old=t['candidates'].setdefault(c['capability_id'],{'capability_id':c['capability_id']})
                for k,v in c.items():
                    if v and not old.get(k): old[k]=v
    return [{**m[s]['stage'],'candidate_l3_capabilities':list(m[s]['candidates'].values())} for s in sorted(m)]


## Production prompt

In [ ]:
SYSTEM_PROMPT='You are performing Level 3 business capability classification. Use Theme Business Needs as primary evidence, Theme Description as supporting context, and the supplied Stage data as the business boundary. Select every supplied candidate L3 directly supported by that context. Use candidate semantic fields when present; do not infer meaning from capability_id alone. Do not select a capability merely because it belongs to the Stage, shares terminology, is broadly related, upstream/downstream, or commonly supports another capability. Return JSON only: {"l3":["CAP00000123"]}. If none are supported, return {"l3":[]}. No reasons, Markdown, or additional fields.'

def theme_context(r):
    return {'theme_business_needs':r['theme_business_needs'],'theme_description':r['theme_description']}

def build_user_prompt(ctx,s,c):
    p={'theme_business_needs':ctx['theme_business_needs'],'theme_description':ctx['theme_description'],'stage_data':s,'candidate_l3_capabilities':c}
    return json.dumps(p,ensure_ascii=False,indent=2)


## Prompt preview — exactly as sent

In [ ]:
r=evaluation_population.iloc[0].to_dict(); p=row_stages(r)[0]
preview_user_prompt=build_user_prompt(theme_context(r),p['stage'],p['candidates'])
print('SYSTEM PROMPT\n'+'='*80+'\n'+SYSTEM_PROMPT+'\n\nFORMATTED USER PROMPT\n'+'='*80+'\n'+preview_user_prompt)

## Prediction and evaluation

In [ ]:
def validate(payload,allowed):
    if not isinstance(payload,dict) or set(payload)!={'l3'} or not isinstance(payload['l3'],list): raise ValueError('Expected {"l3": [...]}')
    out=[]; seen=set(); allowed=set(allowed)
    for cid in payload['l3']:
        if not isinstance(cid,str) or cid.strip() not in allowed or cid.strip() in seen: raise ValueError(f'Invalid L3: {cid}')
        cid=cid.strip(); seen.add(cid); out.append(cid)
    return out

def predict(gateway,ctx,p):
    u=build_user_prompt(ctx,p['stage'],p['candidates'])
    if not p['candidates']: return [],None
    raw,m=call_llm_with_metrics(gateway,SYSTEM_PROMPT,u,id='9zdn8n',reasoning_effort='low')
    return validate(parse_json_response(raw),[c['capability_id'] for c in p['candidates']]),m

def run():
    g=load_gateway(); rr=[]; cc=[]
    for r in evaluation_population.to_dict('records'):
        pred=set(); status='ok'; err=None
        for p in row_stages(r):
            sid=p['stage']['stage_id']; t=perf_counter()
            try:
                vals,m=predict(g,theme_context(r),p); pred.update(vals)
                cc.append({'experiment':EXPERIMENT_NAME,'theme_key':r['theme_key'],'record_key':r['record_key'],'stage_id':sid,'status':'ok','latency_seconds':m.get('latency_seconds') if m else 0,'input_tokens':m.get('input_tokens') if m else 0,'output_tokens':m.get('output_tokens') if m else 0,'total_tokens':m.get('total_tokens') if m else 0,'error':None})
            except Exception as e:
                status='error'; err=str(e); cc.append({'experiment':EXPERIMENT_NAME,'theme_key':r['theme_key'],'record_key':r['record_key'],'stage_id':sid,'status':'error','latency_seconds':perf_counter()-t,'input_tokens':None,'output_tokens':None,'total_tokens':None,'error':err}); break
        truth=r['gt_l3_ids']; scores=score_sets(sorted(pred),truth) if status=='ok' else {'exact_match':None,'precision':None,'recall':None,'f1':None,'predicted_count':None,'truth_count':len(truth)}
        rr.append({'experiment':EXPERIMENT_NAME,'theme_key':r['theme_key'],'record_key':r['record_key'],'stage_ids':as_list(r['stage_ids']),'predicted_l3_ids':sorted(pred) if status=='ok' else None,'gt_l3_ids':truth,'status':status,'error':err,**scores})
    return pd.DataFrame(rr),pd.DataFrame(cc)

results,call_metrics=run(); scored=results[results.status.eq('ok')]; calls=call_metrics[call_metrics.status.eq('ok')]
summary=pd.DataFrame([{'scope':'fixed_50_from_golden_valid_set_seed_42','evaluated_records':len(scored),'exact_match_accuracy':scored.exact_match.mean() if len(scored) else 0,'mean_precision':scored.precision.mean() if len(scored) else 0,'mean_recall':scored.recall.mean() if len(scored) else 0,'mean_f1':scored.f1.mean() if len(scored) else 0}])
latency_tokens=pd.DataFrame([{'successful_calls':len(calls),'failed_calls':int(call_metrics.status.eq('error').sum()),'avg_latency_seconds':calls.latency_seconds.mean() if len(calls) else None,'p50_latency_seconds':calls.latency_seconds.quantile(.5) if len(calls) else None,'p95_latency_seconds':calls.latency_seconds.quantile(.95) if len(calls) else None,'total_input_tokens':calls.input_tokens.sum() if len(calls) else 0,'total_output_tokens':calls.output_tokens.sum() if len(calls) else 0,'total_tokens':calls.total_tokens.sum() if len(calls) else 0,'tokens_per_scored_record':calls.total_tokens.sum()/len(scored) if len(scored) else None}])
display(summary); display(latency_tokens); display(call_metrics); display(results.head(50))
print('Saved',save_results_excel(results,EXPERIMENT_NAME,'results',extra_sheets={'evaluation_summary':summary,'llm_metrics':call_metrics,'latency_tokens':latency_tokens,'evaluation_population':evaluation_population}))
